<a href="https://colab.research.google.com/github/cbonnin88/EcoFlux/blob/main/Data_Generation_EcoFlux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import polars as pl
import numpy as np
from datetime import datetime, timedelta
import plotly.express as px

# **1. Setup Constants**

In [16]:
num_users = 3000
days= 30
start_date = datetime(2026,2,1)
countries = ['France','Denmark','Luxembourg','Germany','Spain','United Kingdom']
plans = ['Free','Basic','Premium']

# **2. Generate User Dimension**

In [17]:
users_ids = [f'user_{i}' for i in range(num_users)]
users_df = pl.DataFrame({
    'user_id': users_ids,
    'signup_date': [start_date + timedelta(days=np.random.randint(0,10)) for _ in range(num_users)],
    'country': np.random.choice(countries, num_users),
    'subscription_plan': np.random.choice(plans, num_users, p=[0.1,0.6,0.3]),
    'device_type': np.random.choice(['iOS','Android'], num_users)
})

# **3. Generate Amplitude Events (Behavioral)**

In [18]:
# Create a skeleton of days
days_df = pl.DataFrame({"day_offset": np.arange(days)})

# Cross-join users with days to create a row for every user-day possibility (90k rows)
events_df = users_df.join(days_df, how="cross")

# Calculate timestamps for every row at once
events_df = events_df.with_columns([
    (pl.col("signup_date") + pl.duration(days=pl.col("day_offset"))).alias("event_date")
])

# Filter out events that happen before signup
events_df = events_df.filter(pl.col("event_date") >= pl.col("signup_date"))

# Vectorized "App Open" generation
# We assign a random "activity_score" to decide if a user opened the app that day
events_df = events_df.with_columns([
    pl.lit(np.random.rand(len(events_df))).alias("activity_score"),
    (pl.col("event_date").cast(pl.Int64) // 1000).alias("time") # Amplitude ms format
])

# Only keep rows where activity_score > 0.3 (Simulating 70% daily activity)
amplitude_events = events_df.filter(pl.col("activity_score") > 0.3).select([
    pl.col("user_id"),
    pl.lit("App Open").alias("event_type"),
    pl.col("time"),
    pl.col("country"),
    pl.struct([
        pl.lit("2.1.0").alias("version"),
        pl.col("subscription_plan")
    ]).alias("event_properties")
])

# **4. Save for Airbyte/BigQuery**

In [19]:
users_df.write_csv('ecoflux_users.csv')
events_df.write_csv('ecoflux_amplitude_events.csv')

print(f'Generated {len(users_df)} users and {len(amplitude_events)} events.')

Generated 3000 users and 63205 events.
